# 03 — DG38 DINOv2 ViT-B/14 Fine-Tune Blocks Ablation (`0`, `2`, `4`, `12` Blocks)

**Mục tiêu (Giai đoạn 6 — Hạn mức đề xuất: 15 CU):**
So sánh định lượng ảnh hưởng của số block Transformer cuối được mở khoá (unfreeze) trên cùng mô hình **DINOv2 ViT-B/14** (`facebook/dinov2-base`), cùng head `LayerNorm(768) -> Dropout(0.2) -> Linear(768, 10)`, cùng manifest cố định (`9,807` Train / `1,226` Val / `1,226` Test):
- **0 blocks (Linear Probe)**: Đóng băng toàn bộ backbone, chỉ train classification head + layernorm (`9,226` tham số, `0.01%`).
- **2 blocks cuối**: Mở khoá `encoder.layer.10..11` + `layernorm` (`14,184,970` tham số, `16.38%`).
- **4 blocks cuối (Cấu hình chính của đồ án)**: Mở khoá `encoder.layer.8..11` + `layernorm` (`28,360,714` tham số, `32.75%`).
- **12 blocks (Full Fine-tune)**: Mở khoá toàn bộ 12 block Transformer (`86,589,706` tham số, `100.00%`) *(chỉ chạy nếu dự báo CU nằm trong hạn mức)*.

Ghi nhận đầy đủ: **Trainable params, % params, thời gian/epoch, Peak VRAM, Validation Macro F1, Test Accuracy & Test Macro F1**.

In [ ]:
# [CELL 1] Thiết lập tham số Fine-Tune Blocks Ablation
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

DRIVE_ROOT = Path(os.environ.get("DG38_DRIVE_ROOT", "/content/drive/MyDrive/DG38_Colab_Package"))
REPO_ROOT = DRIVE_ROOT / "code" if (DRIVE_ROOT / "code" / "model.py").exists() else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EXP_DIR = REPO_ROOT / "evaluation_runs" / "dg38_additional_experiments"
MANIFEST_PATH = EXP_DIR / "manifests" / "classification_train_val_manifest.json"
DATA_ROOT = Path(os.environ.get("DG38_CLS_DATA_ROOT", "/content/dg38_data/classification/standardized_256"))
ARTIFACTS_DIR = (DRIVE_ROOT / "artifacts") if DRIVE_ROOT.exists() else EXP_DIR

BLOCKS_TO_RUN = [0, 2, 4]        # Có thể thêm 12: [0, 2, 4, 12] nếu dự báo CU cho phép
SEED = 42
EPOCHS = 15
OBSERVED_CU_PER_HOUR = None      # <-- BẮT BUỘC: Nhập Usage Rate thực tế từ giao diện Colab (ví dụ: 1.48)
RUN_FULL_ABLATION = False        # Đổi thành True sau khi kiểm tra bảng dự báo CU ở Cell 2

if OBSERVED_CU_PER_HOUR is None or float(OBSERVED_CU_PER_HOUR) <= 0:
    raise ValueError("Vui lòng nhập OBSERVED_CU_PER_HOUR (Usage Rate từ 'Runtime -> View resources') trước khi chạy!")

print(f"DATA_ROOT     : {DATA_ROOT} (exists={DATA_ROOT.exists()})")
print(f"BLOCKS_TO_RUN : {BLOCKS_TO_RUN}")

In [ ]:
# [CELL 2] Chạy Pilot cho cấu hình nặng nhất trong BLOCKS_TO_RUN để dự báo CU
trainer_script = EXP_DIR / "dg38_cls_trainer.py"
max_b = max(BLOCKS_TO_RUN)
pilot_out_dir = ARTIFACTS_DIR / "runs" / f"ablation_dinov2_vitb14_blocks{max_b}_seed{SEED}"

if DATA_ROOT.exists():
    cmd_pilot = [
        sys.executable,
        str(trainer_script),
        "--mode", "pilot",
        "--arch", "dinov2_vitb14",
        "--unfreeze-blocks", str(max_b),
        "--seed", str(SEED),
        "--epochs", str(EPOCHS),
        "--data-root", str(DATA_ROOT),
        "--manifest", str(MANIFEST_PATH),
        "--output-dir", str(pilot_out_dir),
        "--cu-per-hour", str(OBSERVED_CU_PER_HOUR),
        "--budget-category", "finetune_blocks_ablation",
    ]
    res = subprocess.run(cmd_pilot, text=True)
    if res.returncode != 0:
        raise RuntimeError("Pilot Fine-tune Ablation gặp lỗi.")
else:
    print(f"[INFO] Không tìm thấy {DATA_ROOT}. Hãy giải nén dg38_cls_train_val.zip ở Notebook 00 trước.")

In [ ]:
# [CELL 3] Thực thi huấn luyện lần lượt các cấu hình Fine-tune Blocks & Lưu bảng tổng hợp
if not RUN_FULL_ABLATION:
    print("[DỪNG AN TOÀN] RUN_FULL_ABLATION đang là False. Hãy kiểm tra dự báo CU ở Cell 2 trước.")
else:
    for b in BLOCKS_TO_RUN:
        out_dir = ARTIFACTS_DIR / "runs" / f"ablation_dinov2_vitb14_blocks{b}_seed{SEED}"
        cmd_train = [
            sys.executable,
            str(trainer_script),
            "--mode", "train",
            "--arch", "dinov2_vitb14",
            "--unfreeze-blocks", str(b),
            "--seed", str(SEED),
            "--epochs", str(EPOCHS),
            "--data-root", str(DATA_ROOT),
            "--manifest", str(MANIFEST_PATH),
            "--output-dir", str(out_dir),
            "--cu-per-hour", str(OBSERVED_CU_PER_HOUR),
            "--budget-category", "finetune_blocks_ablation",
            "--resume",
        ]
        print(f"=== ĐANG HUẤN LUYỆN DINOv2 ViT-B/14 (UNFREEZE_BLOCKS={b}) ===")
        res = subprocess.run(cmd_train, text=True)
        if res.returncode != 0:
            raise RuntimeError(f"Lỗi khi chạy cấu hình blocks={b}")